# Strategy Prototyping

Rapid iteration on strategy ideas before formalizing into a strategy directory.

**Version**: v1.12.0  
**Architecture**: NO WRAPPERS - Direct Zipline/pandas APIs

## Purpose

This notebook helps you:
1. Quickly test trading logic without creating a full strategy directory
2. Prototype initialize() and handle_data() functions
3. Validate hypothesis with minimal setup
4. Iterate on parameters and signals
5. Transition to formal strategy when ready

## Workflow

1. **Configure**: Set bundle, date range, and initial parameters below
2. **Implement**: Define your strategy logic in initialize() and handle_data()
3. **Test**: Run quick backtest to validate hypothesis
4. **Analyze**: Review metrics and equity curve
5. **Formalize**: When satisfied, migrate to strategies/{asset_class}/{strategy_name}/

**Note**: This is for experimentation. Production strategies should use strategies/_template/ structure.

## Setup

In [ ]:
# Add project root to path
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Standard library
import warnings
from datetime import datetime, timedelta

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Zipline direct imports (v1.12.0 NO WRAPPERS)
from zipline import run_algorithm
from zipline.api import (
    order_target_percent,
    symbol,
    record,
    schedule_function,
    date_rules,
    time_rules,
    set_slippage,
    set_commission,
    get_datetime,
)
from zipline.finance import slippage, commission

# Local imports
from lib.paths import get_project_root, get_results_dir
from lib.bundles import list_bundles, load_bundle
from lib.calendars import get_calendar_for_asset_class
from lib.metrics import calculate_metrics

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

print(f"✓ Setup complete")
print(f"  Project root: {get_project_root()}")

## Configuration

Set your strategy parameters here.

In [ ]:
# Strategy Configuration
strategy_name = 'prototype_sma_cross'  # Name for this prototype
asset_symbol = 'SPY'  # Symbol to trade
bundle_name = None  # None = auto-detect from available bundles

# Backtest Configuration
start_date = '2020-01-01'  # Start date (YYYY-MM-DD)
end_date = '2023-12-31'    # End date (YYYY-MM-DD)
capital_base = 100000.0    # Starting capital ($)

# Strategy Parameters (customize for your strategy)
params = {
    'fast_period': 10,      # Fast moving average period
    'slow_period': 30,      # Slow moving average period
    'position_size': 0.95,  # Position size (0.0-1.0)
}

# Display configuration
print(f"Strategy: {strategy_name}")
print(f"Asset: {asset_symbol}")
print(f"Date range: {start_date} to {end_date}")
print(f"Capital: ${capital_base:,.0f}")
print(f"\nParameters:")
for key, value in params.items():
    print(f"  {key}: {value}")

## Bundle Selection

Auto-detect or manually select a data bundle.

In [ ]:
# List available bundles
available_bundles = list_bundles()
print(f"Available bundles: {len(available_bundles)}")

if available_bundles:
    # Display first 10 bundles
    print("\nBundles (first 10):")
    for bundle in sorted(available_bundles)[:10]:
        print(f"  - {bundle}")
    
    # Auto-select bundle if not specified
    if bundle_name is None:
        # Try to find bundle containing the asset symbol
        matching_bundles = [b for b in available_bundles if asset_symbol.lower() in b.lower()]
        
        if matching_bundles:
            bundle_name = matching_bundles[0]
            print(f"\n✓ Auto-selected bundle: {bundle_name}")
        else:
            # Default to first bundle
            bundle_name = sorted(available_bundles)[0]
            print(f"\n⚠ No matching bundle for {asset_symbol}, using: {bundle_name}")
            print(f"  You may need to change asset_symbol above")
    else:
        print(f"\n✓ Using specified bundle: {bundle_name}")
else:
    print("\n⚠ No bundles found!")
    print("  Ingest data first: python scripts/ingest_data.py --source yahoo --assets equities --timeframe daily")

## Strategy Implementation

Define your trading logic here using Zipline's initialize() and handle_data() pattern.

In [ ]:
def initialize(context):
    """
    Initialize strategy (called once at start).
    
    Example: Simple moving average crossover strategy.
    - Buy when fast MA crosses above slow MA
    - Sell when fast MA crosses below slow MA
    """
    # Set the asset to trade
    context.asset = symbol(asset_symbol)
    
    # Store parameters in context
    context.fast_period = params['fast_period']
    context.slow_period = params['slow_period']
    context.position_size = params['position_size']
    
    # Initialize state variables
    context.invested = False
    
    # Set commission and slippage (realistic costs)
    set_commission(commission.PerShare(cost=0.001, min_trade_cost=1.0))
    set_slippage(slippage.VolumeShareSlippage(volume_limit=0.025, price_impact=0.1))
    
    print(f"✓ Strategy initialized: {strategy_name}")
    print(f"  Asset: {asset_symbol}")
    print(f"  Fast MA: {context.fast_period} days")
    print(f"  Slow MA: {context.slow_period} days")


def handle_data(context, data):
    """
    Handle each bar of data (called on each trading day).
    
    This is where your trading logic executes.
    """
    # Get historical prices using Zipline's direct API
    prices = data.history(
        context.asset,
        'close',
        context.slow_period + 1,  # Need enough history for slow MA
        '1d'
    )
    
    # Calculate moving averages using pandas directly (v1.12.0 NO WRAPPERS)
    fast_ma = prices.rolling(context.fast_period).mean().iloc[-1]
    slow_ma = prices.rolling(context.slow_period).mean().iloc[-1]
    
    # Check if we can trade this asset
    if not data.can_trade(context.asset):
        return
    
    # Trading logic: Simple crossover
    current_price = data.current(context.asset, 'close')
    
    # Generate signals
    if fast_ma > slow_ma and not context.invested:
        # Golden cross - buy signal
        order_target_percent(context.asset, context.position_size)
        context.invested = True
        record(
            signal=1,
            fast_ma=fast_ma,
            slow_ma=slow_ma,
            price=current_price
        )
    elif fast_ma < slow_ma and context.invested:
        # Death cross - sell signal
        order_target_percent(context.asset, 0.0)
        context.invested = False
        record(
            signal=-1,
            fast_ma=fast_ma,
            slow_ma=slow_ma,
            price=current_price
        )
    else:
        # No signal
        record(
            signal=0,
            fast_ma=fast_ma,
            slow_ma=slow_ma,
            price=current_price
        )


print("✓ Strategy functions defined")
print("  - initialize()")
print("  - handle_data()")

## Run Backtest

Execute the backtest using your strategy logic.

In [ ]:
if bundle_name:
    print(f"Running backtest...")
    print(f"  Strategy: {strategy_name}")
    print(f"  Bundle: {bundle_name}")
    print(f"  Date range: {start_date} to {end_date}")
    print(f"  Capital: ${capital_base:,.0f}")
    print()
    
    try:
        # Run backtest using Zipline's direct API (v1.12.0 NO WRAPPERS)
        perf = run_algorithm(
            start=pd.Timestamp(start_date, tz='UTC'),
            end=pd.Timestamp(end_date, tz='UTC'),
            initialize=initialize,
            handle_data=handle_data,
            capital_base=capital_base,
            bundle=bundle_name,
        )
        
        print("\n✓ Backtest complete")
        print(f"  Total days: {len(perf)}")
        print(f"  Date range: {perf.index[0]} to {perf.index[-1]}")
        
    except Exception as e:
        print(f"\n✗ Backtest failed: {e}")
        print(f"  Check that:")
        print(f"    - Bundle '{bundle_name}' exists")
        print(f"    - Symbol '{asset_symbol}' is in the bundle")
        print(f"    - Date range has sufficient data")
        raise
else:
    print("⚠ No bundle available. Cannot run backtest.")

## Performance Metrics

Calculate and display key performance metrics.

In [ ]:
if 'perf' in locals() and not perf.empty:
    # Calculate metrics using lib/metrics (v1.12.0)
    metrics = calculate_metrics(perf)
    
    print("="*60)
    print("PERFORMANCE METRICS")
    print("="*60)
    
    # Returns
    print(f"\nReturns:")
    print(f"  Total Return:        {metrics.get('total_return', 0):.2%}")
    print(f"  Annual Return:       {metrics.get('annual_return', 0):.2%}")
    print(f"  Annual Volatility:   {metrics.get('annual_volatility', 0):.2%}")
    
    # Risk-adjusted metrics
    print(f"\nRisk-Adjusted:")
    print(f"  Sharpe Ratio:        {metrics.get('sharpe', 0):.3f}")
    print(f"  Sortino Ratio:       {metrics.get('sortino', 0):.3f}")
    print(f"  Calmar Ratio:        {metrics.get('calmar', 0):.3f}")
    
    # Risk metrics
    print(f"\nRisk:")
    print(f"  Max Drawdown:        {metrics.get('max_drawdown', 0):.2%}")
    print(f"  Max Drawdown Days:   {metrics.get('max_drawdown_days', 0):.0f}")
    
    # Trade metrics (if available)
    if 'trade_count' in metrics and metrics['trade_count'] > 0:
        print(f"\nTrade Statistics:")
        print(f"  Total Trades:        {metrics.get('trade_count', 0)}")
        print(f"  Win Rate:            {metrics.get('win_rate', 0):.2%}")
        print(f"  Profit Factor:       {metrics.get('profit_factor', 0):.3f}")
        print(f"  Avg Win:             {metrics.get('avg_win', 0):.2%}")
        print(f"  Avg Loss:            {metrics.get('avg_loss', 0):.2%}")
    
    print("="*60)
else:
    print("⚠ No backtest results available")

## Visualizations

Plot equity curve, drawdown, and signals.

In [ ]:
if 'perf' in locals() and not perf.empty:
    # Create figure with subplots
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # 1. Equity Curve
    ax1.plot(perf.index, perf['portfolio_value'], linewidth=1.5, label='Portfolio Value')
    ax1.axhline(y=capital_base, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
    ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
    ax1.set_title(f'{strategy_name} - Performance', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # 2. Drawdown
    cumulative_returns = (perf['portfolio_value'] / capital_base) - 1
    running_max = np.maximum.accumulate(cumulative_returns + 1)
    drawdown = (cumulative_returns + 1) / running_max - 1
    
    ax2.fill_between(perf.index, drawdown * 100, 0, alpha=0.3, color='red', label='Drawdown')
    ax2.set_ylabel('Drawdown (%)', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Trading Signals (if recorded)
    if 'signal' in perf.columns:
        # Plot price and signals
        if 'price' in perf.columns:
            ax3.plot(perf.index, perf['price'], linewidth=1, alpha=0.5, label='Price')
        
        # Mark buy/sell signals
        buy_signals = perf[perf['signal'] == 1]
        sell_signals = perf[perf['signal'] == -1]
        
        if not buy_signals.empty and 'price' in buy_signals.columns:
            ax3.scatter(buy_signals.index, buy_signals['price'], 
                       marker='^', color='green', s=100, alpha=0.7, label='Buy')
        
        if not sell_signals.empty and 'price' in sell_signals.columns:
            ax3.scatter(sell_signals.index, sell_signals['price'], 
                       marker='v', color='red', s=100, alpha=0.7, label='Sell')
        
        # Plot moving averages if recorded
        if 'fast_ma' in perf.columns and 'slow_ma' in perf.columns:
            ax3.plot(perf.index, perf['fast_ma'], linewidth=1, alpha=0.7, 
                    label=f'Fast MA ({params["fast_period"]})', linestyle='--')
            ax3.plot(perf.index, perf['slow_ma'], linewidth=1, alpha=0.7, 
                    label=f'Slow MA ({params["slow_period"]})', linestyle='--')
        
        ax3.set_ylabel('Price ($)', fontsize=12)
        ax3.set_xlabel('Date', fontsize=12)
        ax3.legend(loc='best')
        ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizations generated")
else:
    print("⚠ No backtest results to visualize")

## Returns Analysis

Analyze daily returns distribution.

In [ ]:
if 'perf' in locals() and not perf.empty:
    # Calculate daily returns
    returns = perf['returns']
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax1.hist(returns.dropna(), bins=50, alpha=0.7, edgecolor='black')
    ax1.axvline(returns.mean(), color='red', linestyle='--', 
                label=f'Mean: {returns.mean():.4f}')
    ax1.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.set_xlabel('Daily Returns', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.set_title('Returns Distribution', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Q-Q Plot (normality check)
    from scipy import stats
    stats.probplot(returns.dropna(), dist="norm", plot=ax2)
    ax2.set_title('Q-Q Plot (Normality Check)', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nReturns Statistics:")
    print(f"  Mean:         {returns.mean():.4f}")
    print(f"  Std Dev:      {returns.std():.4f}")
    print(f"  Skewness:     {returns.skew():.4f}")
    print(f"  Kurtosis:     {returns.kurtosis():.4f}")
    print(f"  Min:          {returns.min():.4f}")
    print(f"  Max:          {returns.max():.4f}")
    print(f"  Positive days: {(returns > 0).sum()} ({(returns > 0).sum()/len(returns)*100:.1f}%)")
    print(f"  Negative days: {(returns < 0).sum()} ({(returns < 0).sum()/len(returns)*100:.1f}%)")
else:
    print("⚠ No backtest results available")

## Summary & Next Steps

Review your prototype results and decide next steps.

In [ ]:
print("="*60)
print("STRATEGY PROTOTYPE SUMMARY")
print("="*60)

if 'perf' in locals() and 'metrics' in locals():
    print(f"\nStrategy: {strategy_name}")
    print(f"Asset: {asset_symbol}")
    print(f"Bundle: {bundle_name}")
    print(f"Period: {start_date} to {end_date}")
    
    print(f"\nKey Metrics:")
    print(f"  Total Return:  {metrics.get('total_return', 0):.2%}")
    print(f"  Sharpe Ratio:  {metrics.get('sharpe', 0):.3f}")
    print(f"  Max Drawdown:  {metrics.get('max_drawdown', 0):.2%}")
    
    # Hypothesis validation
    print(f"\nHypothesis Validation:")
    sharpe_ok = metrics.get('sharpe', 0) > 0.5
    dd_ok = abs(metrics.get('max_drawdown', -1)) < 0.3
    return_ok = metrics.get('total_return', -1) > 0
    
    print(f"  Sharpe > 0.5:       {'✓' if sharpe_ok else '✗'} ({metrics.get('sharpe', 0):.3f})")
    print(f"  Max DD < 30%:       {'✓' if dd_ok else '✗'} ({metrics.get('max_drawdown', 0):.2%})")
    print(f"  Positive return:    {'✓' if return_ok else '✗'} ({metrics.get('total_return', 0):.2%})")
    
    # Recommendation
    if sharpe_ok and dd_ok and return_ok:
        print(f"\n✓ Strategy shows promise! Consider:")
        print(f"  1. Formalizing into strategies/{asset_symbol.lower()}/")
        print(f"  2. Parameter optimization (notebooks/02_optimize.ipynb)")
        print(f"  3. Walk-forward validation (notebooks/05_walkforward.ipynb)")
    else:
        print(f"\n⚠ Strategy needs refinement:")
        print(f"  1. Adjust parameters above and re-run")
        print(f"  2. Try different entry/exit logic")
        print(f"  3. Consider additional filters or risk controls")
    
    print(f"\nNext Steps:")
    print(f"  • Iterate on strategy logic in cells above")
    print(f"  • Test different parameter combinations")
    print(f"  • When satisfied, migrate to strategies/_template/")
    print(f"  • Document hypothesis in hypothesis.md")
    print(f"  • Run full validation suite (optimize, walk-forward)")
    
else:
    print("\n⚠ No backtest results available")
    print("\nTo get started:")
    print("  1. Ensure bundle is ingested (run cells above)")
    print("  2. Adjust configuration if needed")
    print("  3. Run backtest cell")

print("="*60)

## Formalize Strategy (Optional)

When ready, create a formal strategy directory.

In [ ]:
# Uncomment and run this cell to create a formal strategy directory

# from shutil import copytree
# from pathlib import Path

# # Infer asset class from bundle
# if any(x in bundle_name.lower() for x in ['forex', 'eur', 'usd']):
#     asset_class = 'forex'
# elif any(x in bundle_name.lower() for x in ['crypto', 'btc', 'eth']):
#     asset_class = 'crypto'
# else:
#     asset_class = 'equities'

# # Create strategy directory
# strategy_dir = get_project_root() / 'strategies' / asset_class / strategy_name

# if strategy_dir.exists():
#     print(f"⚠ Strategy directory already exists: {strategy_dir}")
# else:
#     # Copy template
#     template_dir = get_project_root() / 'strategies' / '_template'
#     copytree(template_dir, strategy_dir)
#     print(f"✓ Created strategy directory: {strategy_dir}")
#     print(f"\nNext steps:")
#     print(f"  1. Edit {strategy_dir}/hypothesis.md with your hypothesis")
#     print(f"  2. Update {strategy_dir}/parameters.yaml with your parameters")
#     print(f"  3. Implement strategy logic in {strategy_dir}/strategy.py")
#     print(f"  4. Run backtest: python scripts/run_backtest.py --strategy {asset_class}/{strategy_name}")

print("Uncomment the code above to create a formal strategy directory")